# Phase 7: Trust Score Regression Models

Train ML models with train/validation/test split to detect overfitting and data leakage.

**Models:** Linear Regression, Random Forest, Gradient Boosting, XGBoost

**Metrics:** RMSE, MAE, R², Spearman Correlation

**Overfitting Detection:** Compare performance across all three datasets

In [9]:
!pip install statsmodels


In [2]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## 1. Load & Prepare Data

In [3]:
df = pd.read_csv("../data/processed/trust_scored_dataset.csv")
print(f"Dataset shape: {df.shape}")

feature_cols = [
    'review_length', 'sentiment_score', 'sentiment_extreme', 'repetition_ratio',
    'unique_word_ratio', 'exclamation_count', 'question_count',
    'user_review_count', 'user_rating_variance', 'user_avg_rating_deviation',
    'user_review_frequency', 'user_extreme_ratio', 'user_burst_flag',
    'user_product_diversity', 'product_review_count', 'product_rating_variance',
    'product_rating_std', 'product_popularity_log', 'product_user_diversity',
    'days_since_first_review', 'review_density', 'review_time_gap', 'burst_indicator',
    'rating', 'rating_deviation', 'verified', 'helpful_ratio'
]

available_features = [col for col in feature_cols if col in df.columns]
X = df[available_features].fillna(0)
y = df['trust_score']

print(f"Features: {len(available_features)}")
print(f"Target shape: {y.shape}")

Dataset shape: (719967, 29)
Features: 5
Target shape: (719967,)


## 2. Train/Validation/Test Split (60/20/20)

In [4]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

print(f"Train: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Val:   {X_val.shape[0]} ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test:  {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")

Train: 431979 (60.0%)
Val:   143994 (20.0%)
Test:  143994 (20.0%)


## 3. Feature Scaling

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
print("Features scaled")

Features scaled


## 3.5 Feature Correlation Analysis

Analyze feature relationships to detect multicollinearity and understand feature importance.

In [6]:
# 3.5.1 Correlation with Target Variable
target_corr = X_train.corrwith(y_train).abs().sort_values(ascending=False)
target_corr_df = pd.DataFrame({
    'Feature': target_corr.index,
    'Correlation': target_corr.values
})

print("\n" + "="*80)
print("FEATURE CORRELATION WITH TARGET (trust_score)")
print("="*80)
print(target_corr_df.to_string(index=False))
print("="*80)

target_corr_df.to_csv('../results/reports/target_correlation.csv', index=False)
print("\nSaved: results/reports/target_correlation.csv")


FEATURE CORRELATION WITH TARGET (trust_score)
         Feature  Correlation
   helpful_ratio     0.673670
rating_deviation     0.397278
        verified     0.262774
   review_length     0.218719
          rating     0.119257

Saved: results/reports/target_correlation.csv


In [7]:
# 3.5.2 Feature-to-Feature Correlation Matrix
import seaborn as sns

corr_matrix = X_train.corr()

# Find highly correlated pairs (> 0.8)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    print("\n" + "="*80)
    print("HIGH CORRELATION PAIRS (|r| > 0.8) - Potential Multicollinearity")
    print("="*80)
    for feat1, feat2, corr in high_corr_pairs:
        print(f"{feat1:30} <-> {feat2:30} : {corr:6.3f}")
    print("="*80)
else:
    print("\n✅ No high correlation pairs found (|r| > 0.8)")

# Plot correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/feature_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.close()
print("\nSaved: results/figures/feature_correlation_matrix.png")




✅ No high correlation pairs found (|r| > 0.8)

Saved: results/figures/feature_correlation_matrix.png


In [8]:
# 3.5.3 Multicollinearity Detection using VIF (Variance Inflation Factor)
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data['Feature'] = X_train.columns
vif_data['VIF'] = [variance_inflation_factor(X_train.values, i) 
                   for i in range(len(X_train.columns))]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("\n" + "="*80)
print("VARIANCE INFLATION FACTOR (VIF) - Multicollinearity Detection")
print("="*80)
print("VIF Interpretation:")
print("  1-5:   Low multicollinearity (acceptable)")
print("  5-10:  Moderate multicollinearity (caution)")
print("  >10:   High multicollinearity (problematic)")
print("="*80)
print(vif_data.to_string(index=False))
print("="*80)

# Flag problematic features
high_vif = vif_data[vif_data['VIF'] > 10]
if len(high_vif) > 0:
    print("\n⚠️  WARNING: High VIF detected for:")
    for _, row in high_vif.iterrows():
        print(f"  - {row['Feature']:30} VIF = {row['VIF']:.2f}")
    print("\nConsider removing or combining these features.")
else:
    print("\n✅ All features have acceptable VIF values (<10)")

vif_data.to_csv('../results/reports/vif_analysis.csv', index=False)
print("\nSaved: results/reports/vif_analysis.csv")

ModuleNotFoundError: No module named 'statsmodels'

## 4. Evaluation Function

In [ ]:
def evaluate_model(y_true, y_pred, model_name, dataset_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    spearman, _ = spearmanr(y_true, y_pred)
    return {'Model': model_name, 'Dataset': dataset_name, 'RMSE': rmse, 'MAE': mae, 'R2': r2, 'Spearman': spearman}

## 5. Train Models

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_results = [
    evaluate_model(y_train, lr.predict(X_train_scaled), 'Linear Regression', 'Train'),
    evaluate_model(y_val, lr.predict(X_val_scaled), 'Linear Regression', 'Validation'),
    evaluate_model(y_test, lr.predict(X_test_scaled), 'Linear Regression', 'Test')
]
print("Linear Regression trained")

In [ ]:
rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_results = [
    evaluate_model(y_train, rf.predict(X_train), 'Random Forest', 'Train'),
    evaluate_model(y_val, rf.predict(X_val), 'Random Forest', 'Validation'),
    evaluate_model(y_test, rf.predict(X_test), 'Random Forest', 'Test')
]
print("Random Forest trained")

In [ ]:
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
gb.fit(X_train, y_train)
gb_results = [
    evaluate_model(y_train, gb.predict(X_train), 'Gradient Boosting', 'Train'),
    evaluate_model(y_val, gb.predict(X_val), 'Gradient Boosting', 'Validation'),
    evaluate_model(y_test, gb.predict(X_test), 'Gradient Boosting', 'Test')
]
print("Gradient Boosting trained")

In [ ]:
xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
xgb_results = [
    evaluate_model(y_train, xgb.predict(X_train), 'XGBoost', 'Train'),
    evaluate_model(y_val, xgb.predict(X_val), 'XGBoost', 'Validation'),
    evaluate_model(y_test, xgb.predict(X_test), 'XGBoost', 'Test')
]
print("XGBoost trained")

## 5.5 K-Fold Cross-Validation

Perform 5-fold cross-validation to validate model stability and generalization.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

# Setup 5-fold cross-validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Models to evaluate
cv_models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1)
}

print("\n" + "="*100)
print("K-FOLD CROSS-VALIDATION RESULTS (5 Folds)")
print("="*100)
print(f"{'Model':<25} {'Mean R²':>12} {'Std R²':>12} {'Min R²':>12} {'Max R²':>12}")
print("-" * 100)

cv_results = []

for name, model in cv_models.items():
    # Use scaled data for Linear Regression, raw data for tree-based models
    if name == 'Linear Regression':
        X_cv = X_train_scaled
    else:
        X_cv = X_train
    
    # Perform cross-validation
    scores = cross_val_score(model, X_cv, y_train, cv=kfold, 
                            scoring='r2', n_jobs=-1)
    
    cv_results.append({
        'Model': name,
        'Mean_R2': scores.mean(),
        'Std_R2': scores.std(),
        'Min_R2': scores.min(),
        'Max_R2': scores.max(),
        'Fold_Scores': scores
    })
    
    print(f"{name:<25} {scores.mean():>12.4f} {scores.std():>12.4f} "
          f"{scores.min():>12.4f} {scores.max():>12.4f}")

print("="*100)

# Save results
cv_df = pd.DataFrame([{
    'Model': r['Model'],
    'Mean_R2': r['Mean_R2'],
    'Std_R2': r['Std_R2'],
    'Min_R2': r['Min_R2'],
    'Max_R2': r['Max_R2']
} for r in cv_results])

cv_df.to_csv('../results/reports/cross_validation_results.csv', index=False)
print("\nSaved: results/reports/cross_validation_results.csv")

In [ ]:
# Visualize cross-validation results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Mean R² with error bars
models = [r['Model'] for r in cv_results]
means = [r['Mean_R2'] for r in cv_results]
stds = [r['Std_R2'] for r in cv_results]

ax1.barh(models, means, xerr=stds, capsize=5, color='steelblue', alpha=0.7)
ax1.set_xlabel('R² Score', fontweight='bold')
ax1.set_title('Cross-Validation Mean R² (±1 Std)', fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# Plot 2: Box plot of fold scores
fold_data = [r['Fold_Scores'] for r in cv_results]
bp = ax2.boxplot(fold_data, labels=models, vert=False, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightcoral')
    patch.set_alpha(0.7)
ax2.set_xlabel('R² Score', fontweight='bold')
ax2.set_title('Cross-Validation Score Distribution', fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/cross_validation_analysis.png', dpi=300, bbox_inches='tight')
plt.close()
print("\nSaved: results/figures/cross_validation_analysis.png")

## 6. Comprehensive Results

In [ ]:
all_results = pd.DataFrame(lr_results + rf_results + gb_results + xgb_results)

print("\n" + "="*100)
print("ALL MODELS - TRAIN/VALIDATION/TEST PERFORMANCE")
print("="*100)
print(all_results.to_string(index=False))
print("="*100)

all_results.to_csv('../results/reports/model_performance_all_datasets.csv', index=False)

## 7. Overfitting & Data Leakage Analysis

In [ ]:
models = ['Linear Regression', 'Random Forest', 'Gradient Boosting', 'XGBoost']
analysis = []

for model in models:
    m_data = all_results[all_results['Model'] == model]
    train_r2 = m_data[m_data['Dataset'] == 'Train']['R2'].values[0]
    val_r2 = m_data[m_data['Dataset'] == 'Validation']['R2'].values[0]
    test_r2 = m_data[m_data['Dataset'] == 'Test']['R2'].values[0]
    
    train_rmse = m_data[m_data['Dataset'] == 'Train']['RMSE'].values[0]
    val_rmse = m_data[m_data['Dataset'] == 'Validation']['RMSE'].values[0]
    test_rmse = m_data[m_data['Dataset'] == 'Test']['RMSE'].values[0]
    
    train_spear = m_data[m_data['Dataset'] == 'Train']['Spearman'].values[0]
    val_spear = m_data[m_data['Dataset'] == 'Validation']['Spearman'].values[0]
    test_spear = m_data[m_data['Dataset'] == 'Test']['Spearman'].values[0]
    
    r2_gap = train_r2 - test_r2
    rmse_gap = test_rmse - train_rmse
    spear_gap = train_spear - test_spear
    
    analysis.append({
        'Model': model,
        'Train_R2': train_r2, 'Val_R2': val_r2, 'Test_R2': test_r2,
        'R2_Gap': r2_gap,
        'Train_RMSE': train_rmse, 'Val_RMSE': val_rmse, 'Test_RMSE': test_rmse,
        'RMSE_Gap': rmse_gap,
        'Train_Spear': train_spear, 'Val_Spear': val_spear, 'Test_Spear': test_spear,
        'Spear_Gap': spear_gap
    })

analysis_df = pd.DataFrame(analysis)

print("\n" + "="*120)
print("OVERFITTING ANALYSIS - R² METRICS")
print("="*120)
print(analysis_df[['Model', 'Train_R2', 'Val_R2', 'Test_R2', 'R2_Gap']].to_string(index=False))
print("="*120)

print("\n" + "="*120)
print("OVERFITTING ANALYSIS - RMSE METRICS")
print("="*120)
print(analysis_df[['Model', 'Train_RMSE', 'Val_RMSE', 'Test_RMSE', 'RMSE_Gap']].to_string(index=False))
print("="*120)

print("\n" + "="*120)
print("OVERFITTING ANALYSIS - SPEARMAN CORRELATION (Ranking Quality)")
print("="*120)
print(analysis_df[['Model', 'Train_Spear', 'Val_Spear', 'Test_Spear', 'Spear_Gap']].to_string(index=False))
print("="*120)

analysis_df.to_csv('../results/reports/overfitting_analysis.csv', index=False)

## 8. Data Leakage & Overfitting Detection

In [ ]:
print("\n" + "="*120)
print("DATA LEAKAGE & OVERFITTING DETECTION")
print("="*120)

for idx, row in analysis_df.iterrows():
    print(f"\n{row['Model']}:")
    print("-" * 100)
    
    # Check 1: Validation between train and test
    val_between = (row['Val_R2'] <= row['Train_R2']) and (row['Val_R2'] >= row['Test_R2'])
    print(f"  ✓ Val R² between Train and Test: {val_between}")
    print(f"    Train: {row['Train_R2']:.4f}, Val: {row['Val_R2']:.4f}, Test: {row['Test_R2']:.4f}")
    
    # Check 2: Overfitting (R² gap)
    overfitting = row['R2_Gap'] > 0.05
    print(f"  ✓ Overfitting detected (R² gap > 0.05): {overfitting}")
    print(f"    R² Gap (Train - Test): {row['R2_Gap']:.4f}")
    
    # Check 3: RMSE increases from train to test
    rmse_increase = row['Test_RMSE'] >= row['Train_RMSE']
    print(f"  ✓ RMSE increases Train→Test (expected): {rmse_increase}")
    print(f"    Train: {row['Train_RMSE']:.4f}, Test: {row['Test_RMSE']:.4f}")
    
    # Check 4: Spearman consistency
    spear_consistent = row['Spear_Gap'] < 0.05
    print(f"  ✓ Spearman consistent (gap < 0.05): {spear_consistent}")
    print(f"    Train: {row['Train_Spear']:.4f}, Test: {row['Test_Spear']:.4f}")
    
    # Check 5: Val-Test gap small
    val_test_gap = abs(row['Val_R2'] - row['Test_R2'])
    val_proxy = val_test_gap < 0.03
    print(f"  ✓ Val is good proxy for Test (gap < 0.03): {val_proxy}")
    print(f"    Val-Test R² gap: {val_test_gap:.4f}")
    
    # Overall health
    checks = [val_between, not overfitting, rmse_increase, spear_consistent, val_proxy]
    health = sum(checks) / len(checks) * 100
    print(f"\n  Model Health Score: {health:.0f}% ({sum(checks)}/5 checks passed)")
    
    if health >= 80:
        print(f"  Status: ✅ HEALTHY - No significant issues")
    elif health >= 60:
        print(f"  Status: ⚠️  WARNING - Minor issues detected")
    else:
        print(f"  Status: ❌ CRITICAL - Significant overfitting/leakage")

print("\n" + "="*120)

## 9. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

models_list = analysis_df['Model'].tolist()
x = np.arange(len(models_list))
width = 0.25

# R² comparison
axes[0, 0].bar(x - width, analysis_df['Train_R2'], width, label='Train', color='green', alpha=0.7)
axes[0, 0].bar(x, analysis_df['Val_R2'], width, label='Validation', color='orange', alpha=0.7)
axes[0, 0].bar(x + width, analysis_df['Test_R2'], width, label='Test', color='red', alpha=0.7)
axes[0, 0].set_ylabel('R² Score', fontsize=12)
axes[0, 0].set_title('R² Across Datasets', fontsize=14, fontweight='bold')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(models_list, rotation=15, ha='right')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# RMSE comparison
axes[0, 1].bar(x - width, analysis_df['Train_RMSE'], width, label='Train', color='green', alpha=0.7)
axes[0, 1].bar(x, analysis_df['Val_RMSE'], width, label='Validation', color='orange', alpha=0.7)
axes[0, 1].bar(x + width, analysis_df['Test_RMSE'], width, label='Test', color='red', alpha=0.7)
axes[0, 1].set_ylabel('RMSE', fontsize=12)
axes[0, 1].set_title('RMSE Across Datasets', fontsize=14, fontweight='bold')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(models_list, rotation=15, ha='right')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# Spearman comparison
axes[1, 0].bar(x - width, analysis_df['Train_Spear'], width, label='Train', color='green', alpha=0.7)
axes[1, 0].bar(x, analysis_df['Val_Spear'], width, label='Validation', color='orange', alpha=0.7)
axes[1, 0].bar(x + width, analysis_df['Test_Spear'], width, label='Test', color='red', alpha=0.7)
axes[1, 0].set_ylabel('Spearman Correlation', fontsize=12)
axes[1, 0].set_title('Spearman Correlation Across Datasets', fontsize=14, fontweight='bold')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(models_list, rotation=15, ha='right')
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# R² Gap (overfitting severity)
colors = ['green' if x < 0.05 else 'orange' if x < 0.1 else 'red' for x in analysis_df['R2_Gap']]
axes[1, 1].bar(models_list, analysis_df['R2_Gap'], color=colors, alpha=0.7)
axes[1, 1].set_ylabel('R² Gap (Train - Test)', fontsize=12)
axes[1, 1].set_title('Overfitting Severity', fontsize=14, fontweight='bold')
axes[1, 1].set_xticklabels(models_list, rotation=15, ha='right')
axes[1, 1].axhline(y=0.05, color='orange', linestyle='--', linewidth=2, label='Warning')
axes[1, 1].axhline(y=0.1, color='red', linestyle='--', linewidth=2, label='Critical')
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/overfitting_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved")

## 9.5 Ablation Study

Measure the impact of each feature by removing them one at a time.

In [ ]:
# Define feature categories for ablation
# NOTE: Only 5 features are currently available
feature_categories = {
    'review_length': ['review_length'],
    'rating': ['rating'],
    'rating_deviation': ['rating_deviation'],
    'verified': ['verified'],
    'helpful_ratio': ['helpful_ratio']
}

print("Feature Categories for Ablation Study:")
for category, features in feature_categories.items():
    print(f"  {category}: {features}")

In [ ]:
# Perform ablation study
print("\n" + "="*100)
print("ABLATION STUDY - Feature Impact Analysis")
print("="*100)
print("Removing each feature and measuring performance degradation...\n")

ablation_results = []

# Baseline performance (all features)
baseline_r2 = xgb.score(X_test, y_test)
baseline_spearman, _ = spearmanr(y_test, xgb.predict(X_test))

print(f"Baseline (All Features): R² = {baseline_r2:.4f}, Spearman = {baseline_spearman:.4f}\n")

# Test each feature removal
for category, features_to_remove in feature_categories.items():
    # Create dataset without this feature
    remaining_features = [f for f in available_features if f not in features_to_remove]
    
    if len(remaining_features) == 0:
        continue
    
    X_train_ablation = X_train[remaining_features]
    X_test_ablation = X_test[remaining_features]
    
    # Train XGBoost without this feature
    model_ablation = XGBRegressor(n_estimators=100, learning_rate=0.1, 
                                  max_depth=6, random_state=42, n_jobs=-1)
    model_ablation.fit(X_train_ablation, y_train)
    
    # Evaluate
    r2_ablation = model_ablation.score(X_test_ablation, y_test)
    spearman_ablation, _ = spearmanr(y_test, model_ablation.predict(X_test_ablation))
    
    # Calculate degradation
    r2_degradation = ((baseline_r2 - r2_ablation) / baseline_r2) * 100
    spearman_degradation = ((baseline_spearman - spearman_ablation) / baseline_spearman) * 100
    
    ablation_results.append({
        'Feature_Removed': category,
        'R2_Without': r2_ablation,
        'R2_Degradation_%': r2_degradation,
        'Spearman_Without': spearman_ablation,
        'Spearman_Degradation_%': spearman_degradation
    })
    
    print(f"Without {category:20} → R² = {r2_ablation:.4f} ({r2_degradation:+6.2f}%), "
          f"Spearman = {spearman_ablation:.4f} ({spearman_degradation:+6.2f}%)")

print("="*100)

In [ ]:
# Ablation Summary & Feature Importance
ablation_df = pd.DataFrame(ablation_results)
ablation_df = ablation_df.sort_values('R2_Degradation_%', ascending=False)

print("\n" + "="*100)
print("ABLATION STUDY SUMMARY - Features Ranked by Importance")
print("="*100)
print(ablation_df.to_string(index=False))
print("="*100)

# Identify critical features
critical_features = ablation_df[ablation_df['R2_Degradation_%'] > 5]
if len(critical_features) > 0:
    print("\n🔴 CRITICAL FEATURES (>5% performance drop when removed):")
    for _, row in critical_features.iterrows():
        print(f"  - {row['Feature_Removed']:20} → {row['R2_Degradation_%']:6.2f}% degradation")
else:
    print("\n✅ No single feature causes >5% performance drop")

# Save results
ablation_df.to_csv('../results/reports/ablation_study.csv', index=False)
print("\nSaved: results/reports/ablation_study.csv")

In [ ]:
# Ablation Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: R² Degradation
ax1.barh(ablation_df['Feature_Removed'], ablation_df['R2_Degradation_%'], 
         color='crimson', alpha=0.7)
ax1.set_xlabel('R² Degradation (%)', fontweight='bold')
ax1.set_ylabel('Feature Removed', fontweight='bold')
ax1.set_title('Feature Importance - R² Impact', fontweight='bold')
ax1.axvline(x=5, color='red', linestyle='--', linewidth=2, label='Critical Threshold (5%)')
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

# Plot 2: Spearman Degradation
ax2.barh(ablation_df['Feature_Removed'], ablation_df['Spearman_Degradation_%'], 
         color='darkorange', alpha=0.7)
ax2.set_xlabel('Spearman Degradation (%)', fontweight='bold')
ax2.set_ylabel('Feature Removed', fontweight='bold')
ax2.set_title('Feature Importance - Ranking Impact', fontweight='bold')
ax2.axvline(x=5, color='red', linestyle='--', linewidth=2, label='Critical Threshold (5%)')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/ablation_analysis.png', dpi=300, bbox_inches='tight')
plt.close()
print("\nSaved: results/figures/ablation_analysis.png")

## 10. Save Best Model

In [ ]:
# Select best model based on test set Spearman
test_results = all_results[all_results['Dataset'] == 'Test'].sort_values('Spearman', ascending=False)
best_model_name = test_results.iloc[0]['Model']

if best_model_name == 'XGBoost':
    best_model = xgb
elif best_model_name == 'Random Forest':
    best_model = rf
elif best_model_name == 'Gradient Boosting':
    best_model = gb
else:
    best_model = lr

joblib.dump(best_model, '../models/trained/best_trust_model.pkl')
joblib.dump(scaler, '../models/feature_scaler.pkl')

with open('../models/trained/feature_names.txt', 'w') as f:
    f.write('\n'.join(available_features))

print(f"Best model: {best_model_name}")
print(f"Test Spearman: {test_results.iloc[0]['Spearman']:.4f}")
print("Model saved")

## Summary

**Phase 7 Complete:**
- Trained 4 models with train/validation/test split
- Comprehensive overfitting analysis across all datasets
- Data leakage detection through validation consistency
- Best model selected based on test set performance
- All metrics saved for Phase 8 aggregation